In [1]:
import sys
sys.path.append('..')

In [2]:
from utils.prompts import render
from utils.router import pick_model
from utils.llm_client import LLMClient
from utils.config_loader import reload_config
from pathlib import Path
reload_config()

In [3]:
scenario_path = Path("../data/Scenarios.txt")

with open(scenario_path, "r", encoding="utf-8") as file:
    scenario_data = file.read()

scenarios = [
    scenario.strip()
    for scenario in scenario_data.split("\n\n")
    if scenario.strip()
]

print(f"Loaded {len(scenarios)} scenarios.")

Loaded 2 scenarios.


In [4]:
def analyze_scenario(scenario, llm, temperature):

    prompt_text, _ = render(
        "cot_reasoning.v1",
        role="crisis response decision assistant",
        query=scenario,
        instruction="""
Analyze the scenario in stages:

1. Identify all urgent problems mentioned in the scenario.
2. For each problem, identify who or what is at risk.
3. Determine the severity and time-sensitivity of each risk using only the given information.
4. Compare the competing risks.
5. Identify which problem requires action first.
6. Give the final recommended priority with a brief justification.
""",
        constraints="""
Use only information explicitly provided in the scenario.
Do not invent rescue teams, vehicles, medical resources, travel times,
victim counts, or equipment.

If important information is missing, state that it is unknown.
""",
        format="""
Urgent Issues: [identified issues]
Risk Comparison: [brief comparison]
Recommended Priority: [action]
Reason: [brief explanation]
"""
    )

    response = llm.chat(
        [
            {
                "role": "user",
                "content": prompt_text
            }
        ],
        temperature=temperature,
        task_type="reasoning"
    )

    return response["text"].strip()

In [5]:
model = pick_model(
    provider="groq",
    technique="cot_reasoning"
)

print(model)

llm = LLMClient(
    "groq",
    model
)

openai/gpt-oss-120b


In [6]:
experiment_results = []

for scenario_index, scenario in enumerate(scenarios, start=1):

    print("=" * 80)
    print(f"SCENARIO {scenario_index}")
    print("=" * 80)

    chaos_outputs = []

    # Chaos Mode
    for run in range(1, 4):

        output = analyze_scenario(
            scenario,
            llm,
            temperature=1.0
        )

        chaos_outputs.append(output)

        print(f"\nCHAOS MODE - RUN {run}")
        print(output)

    # Safe Mode
    safe_output = analyze_scenario(
        scenario,
        llm,
        temperature=0.0
    )

    print("\nSAFE MODE")
    print(safe_output)

    experiment_results.append({
        "scenario": scenario,
        "chaos_outputs": chaos_outputs,
        "safe_output": safe_output
    })

SCENARIO 1

CHAOS MODE - RUN 1
**Urgent Issues:**  
1. Uncle trapped in line rooms, climbing a tree as water rises – immediate life‑threat.  
2. Diabetic patient collapsed inside the factory and needs insulin – immediate health‑threat.  
3. 40 people are hungry – non‑life‑threat (needs care but not immediate).

**Risk Comparison:**  
- The uncle faces drowning or injury from rising water; without rescue he will die quickly.  
- The diabetic patient faces potential severe hypoglycemia or hyperglycemia; without insulin he can deteriorate rapidly and die.  
- Hunger in 40 people is serious but not immediately fatal; they can survive for many hours/days without food.

**Recommended Priority:**  
Rescue the uncle trapped in the line rooms (water‑rising situation).

**Reason:**  
Both the uncle and the diabetic patient are facing immediate, life‑threatening conditions, but the uncle is in a rapidly worsening environment (rising water) that can cause death within minutes, whereas a diabetic c

In [7]:
output_path = Path("../output/stability_experiment.md")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as file:

    file.write("# Part 2 - Temperature Stability Experiment\n\n")

    file.write(
        "This experiment compares CoT reasoning at "
        "`temperature=1.0` (Chaos Mode) and "
        "`temperature=0.0` (Safe Mode).\n\n"
    )

    for index, result in enumerate(experiment_results, start=1):

        file.write(f"## Scenario {index}\n\n")

        file.write("### Input Scenario\n\n")
        file.write(f"{result['scenario']}\n\n")

        file.write("### Chaos Mode - Temperature 1.0\n\n")

        for run_index, output in enumerate(
            result["chaos_outputs"],
            start=1
        ):
            file.write(f"#### Run {run_index}\n\n")
            file.write(f"{output}\n\n")

        file.write("### Safe Mode - Temperature 0.0\n\n")
        file.write(f"{result['safe_output']}\n\n")

        file.write("---\n\n")

print(f"Saved to: {output_path}")

Saved to: ..\output\stability_experiment.md
